In [1]:
from pathlib import Path
# Default Windows root or auto-detected local project root
PROJECT_ROOT = Path(r"F:\PythonFiles\Gemini\NetworkSDE") if (Path(r"F:\PythonFiles\Gemini\NetworkSDE") / "highd_nb" / "api.py").is_file() else Path.cwd().resolve()
# 留空会创建带时间戳的新目录。已存在的合成目录不覆盖。
TEST_ROOT_OVERRIDE = None
DEVICE = "auto"  # NVIDIA CUDA 可执行时优先；否则明确使用 CPU。


In [ ]:
import sys
from datetime import datetime
from IPython.display import display
if not (PROJECT_ROOT / "highd_nb/api.py").is_file():
    raise FileNotFoundError("请检查 PROJECT_ROOT 和解压层级。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
from highd_nb import Workflow
from highd_nb.testing import make_test_data, synthetic_config, run_unit_tests
from highd_nb.core.common import write_json

TEST_ROOT = Path(TEST_ROOT_OVERRIDE) if TEST_ROOT_OVERRIDE else PROJECT_ROOT / "outputs" / ("_software_test_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
TEST_RAW = TEST_ROOT / "synthetic_raw"
TEST_WORK = TEST_ROOT / "synthetic_work"
print("本次独立软件测试目录：", TEST_ROOT)

## 1. 自动单元测试

In [ ]:
unit_result = run_unit_tests(TEST_ROOT / "test_logs")
display(unit_result)

## 2. 生成明确标记的合成数据

In [ ]:
make_test_data(TEST_RAW, n_recordings=4, reproduce_roi_5_07=True)
wf = Workflow(TEST_RAW, TEST_WORK, config=synthetic_config(), allow_synthetic=True)
display(wf.environment())
display(wf.preview("01", 4)["tracks"])

## 3. 完整性与共同区域

In [ ]:
display(wf.audit(expected_recordings=4))
display(wf.preflight())
wf.require_ready()

## 4. 全尺度处理及报告

In [ ]:
display(wf.preprocess(scales=[50,75,100]))
display(wf.report())
plan=wf.plan()

## 5. 两类模型，三尺度，一个留地点折；各2 epochs，仅为程序测试

In [ ]:
display(wf.train(folds="1",scales="all",models="masked,latent",seeds=[7],epochs=2,
                 run_tag="selftest",device=DEVICE))

## 6. 公式提取与测试；允许候选被科学门控拒绝，不允许运行异常被忽略

In [ ]:
display(wf.symbolic(run_tag="selftest",device=DEVICE))
wf.evaluate(run_tag="selftest",device=DEVICE,confirm_protocol_frozen=True)
wf.evaluate(run_tag="selftest",symbolic=True,device=DEVICE,confirm_protocol_frozen=True)
display(wf.summarize("selftest"))

## 7. 保存实际测试完成状态

In [ ]:
wf.collect()
result={"status":"PASS", "data":"SYNTHETIC SOFTWARE TEST ONLY", "real_highD_executed":False,
        "scales_m":[50,75,100], "model_branches":["masked","latent"], "epochs_each":2,
        "outer_test_location":1, "notebook":"00_SelfTest.ipynb"}
write_json(TEST_ROOT / "SELFTEST_RESULT.json",result)
display(result)